# 핸즈온 01 — 모델 × Thinking Level 비용·성능 매트릭스

**소요 시간**: 60~70분
**학습 목표**:
1. Gemini 3 시리즈 3개 모델 × 3개 thinking level = **9개 조합**을 동일 입력으로 호출한다
2. **latency, 입력 토큰, 출력 토큰, thinking 토큰, 추정 비용**을 측정한다
3. ITS 도메인 워크로드별로 어떤 조합이 적합한지 의사결정 프레임을 만든다

> **왜 이 핸즈온이 첫 정량적 실습인가?**
> 모델 라인업과 thinking_level의 비용 영향은 **숫자로 봐야** 체감됩니다. 강의 자료에서 "비용 가치 중심 설계"를 강조했지만, 실제 측정 없이는 설득력이 약합니다.

## 1-1. 환경 셋업

In [ ]:
!pip install -q -U google-genai pandas matplotlib pillow requests

In [ ]:
import os, time, json
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("✅ Client ready")

### 한글 폰트 (matplotlib에서 한글 깨짐 방지)

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm

# 나눔고딕 등록 (Colab 재시작 필요할 수 있음)
font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rc("axes", unicode_minus=False)
print("Font:", plt.rcParams["font.family"])

## 1-2. 테스트 입력 준비 — 도로 정체 이미지

청중이 ITS 회사이므로, 분석 대상은 **도로 정체 사진**으로 통일합니다. 공개 라이선스 이미지를 Wikimedia Commons에서 가져옵니다.

> 강의 당일 네트워크 이슈가 있다면, 강사가 사전에 다운받아둔 이미지를 사용합니다 (강사 가이드 참조).

In [ ]:
import requests
from PIL import Image
from io import BytesIO

# Wikimedia Commons - 공개 라이선스 도로 정체 이미지
# (강의 당일 다른 URL이 필요하면 강사 가이드 참조)
IMG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/9/97/I-80_Eastshore_Freeway.jpg/1280px-I-80_Eastshore_Freeway.jpg"

def fetch_image(url):
    r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    img = Image.open(BytesIO(r.content)).convert("RGB")
    # 토큰 절약을 위해 1024px로 리사이즈
    img.thumbnail((1024, 1024))
    return img

img = fetch_image(IMG_URL)
print(f"Image size: {img.size}")
img

### 테스트 프롬프트 정의

비교가 유의미하려면 **변수를 통제**해야 합니다. 모든 호출에 동일한 이미지 + 동일한 프롬프트를 사용합니다.

In [ ]:
PROMPT = """이 도로 정체 사진을 ITS 운영자 관점에서 분석해주세요.

다음을 JSON으로 출력하세요:
{
  "vehicle_count_estimate": <대략적인 차량 대수>,
  "congestion_level": "low" | "moderate" | "heavy",
  "primary_cause": <정체의 주된 원인 추정>,
  "weather_or_time": <날씨 또는 시간대 단서>,
  "recommended_action": <ITS 센터에서 취할 수 있는 조치 1가지>
}

순수 JSON만 출력하고 다른 설명은 생략하세요."""

print(PROMPT)

## 1-3. 9개 조합 측정 함수

In [ ]:
# 2026년 5월 기준 가격 (per 1M tokens, ≤200K context)
# 출처: https://ai.google.dev/gemini-api/docs/pricing
PRICING = {
    "gemini-3.1-pro-preview":         {"input": 2.00, "output": 12.00},
    "gemini-3-flash-preview":         {"input": 0.50, "output":  3.00},
    "gemini-3.1-flash-lite-preview":  {"input": 0.25, "output":  1.50},
}

def call_with_config(model, thinking_level, prompt, image):
    """모델 + thinking_level + 이미지 + 프롬프트로 1회 호출."""
    t0 = time.time()
    config = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level=thinking_level),
        response_mime_type="application/json",  # JSON 강제
    )
    try:
        resp = client.models.generate_content(
            model=model,
            contents=[prompt, image],
            config=config,
        )
        latency = time.time() - t0
        u = resp.usage_metadata
        thinking = getattr(u, "thoughts_token_count", 0) or 0
        # 비용 계산 (output에 thinking 포함)
        p = PRICING[model]
        cost = (
            u.prompt_token_count * p["input"] / 1_000_000
            + (u.candidates_token_count + thinking) * p["output"] / 1_000_000
        )
        return {
            "model": model,
            "thinking": thinking_level,
            "latency_s": round(latency, 2),
            "input_tokens": u.prompt_token_count,
            "output_tokens": u.candidates_token_count,
            "thinking_tokens": thinking,
            "total_tokens": u.total_token_count,
            "cost_usd": round(cost, 6),
            "response": resp.text,
            "error": None,
        }
    except Exception as e:
        return {
            "model": model, "thinking": thinking_level,
            "latency_s": None, "error": str(e)[:80],
            "input_tokens": None, "output_tokens": None,
            "thinking_tokens": None, "total_tokens": None,
            "cost_usd": None, "response": None,
        }


## 1-4. 9개 조합 모두 실행

각 호출 후 1초 대기를 둡니다 (무료 티어 RPM 보호).

In [ ]:
MODELS = [
    "gemini-3.1-pro-preview",
    "gemini-3-flash-preview",
    "gemini-3.1-flash-lite-preview",
]
LEVELS = ["low", "medium", "high"]

results = []
for model in MODELS:
    for level in LEVELS:
        print(f"⏳ {model} / thinking={level}...", end=" ", flush=True)
        r = call_with_config(model, level, PROMPT, img)
        results.append(r)
        if r["error"]:
            print(f"❌ {r['error']}")
        else:
            print(f"✅ {r['latency_s']}s  ${r['cost_usd']:.6f}")
        time.sleep(1.5)  # rate limit 보호

print("\n측정 완료")

## 1-5. 결과 매트릭스 — pandas로 정리

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df_show = df[[
    "model", "thinking", "latency_s",
    "input_tokens", "output_tokens", "thinking_tokens",
    "cost_usd"
]].copy()
df_show["model"] = df_show["model"].str.replace("-preview", "")
df_show.style.format({
    "cost_usd": "${:.6f}",
    "latency_s": "{:.2f}s",
    "input_tokens": "{:,}",
    "output_tokens": "{:,}",
    "thinking_tokens": "{:,}",
})

## 1-6. 시각화 — 비용 vs 지연시간 산점도

이 그래프가 **모델 선택의 핵심 의사결정 도구**입니다.

In [ ]:
import matplotlib.pyplot as plt

# 성공한 결과만
df_ok = df[df["error"].isna()].copy()
df_ok["model_short"] = df_ok["model"].str.replace("gemini-", "").str.replace("-preview", "")

fig, ax = plt.subplots(figsize=(10, 6))

colors = {"low": "#2ecc71", "medium": "#f39c12", "high": "#e74c3c"}
markers = {
    "3.1-pro": "o",
    "3-flash": "s",
    "3.1-flash-lite": "^",
}

for _, row in df_ok.iterrows():
    ax.scatter(
        row["latency_s"], row["cost_usd"] * 1000,  # mUSD
        c=colors[row["thinking"]],
        marker=markers.get(row["model_short"], "x"),
        s=220, alpha=0.75, edgecolors="black", linewidth=1,
    )
    ax.annotate(
        f"{row['model_short']}\n{row['thinking']}",
        (row["latency_s"], row["cost_usd"] * 1000),
        xytext=(7, 7), textcoords="offset points",
        fontsize=8,
    )

ax.set_xlabel("Latency (초)")
ax.set_ylabel("호출 1회당 비용 (mUSD)")
ax.set_title("Gemini 3 모델 × Thinking Level — 비용 vs 지연시간")
ax.grid(True, alpha=0.3)

# 범례
legend_elements = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=v, markersize=12, label=f"thinking={k}")
    for k, v in colors.items()
]
ax.legend(handles=legend_elements, loc="upper left")
plt.tight_layout()
plt.show()

## 1-7. 응답 품질 비교

비용·속도뿐 아니라 **응답의 정확성**도 봐야 합니다. 9개 조합의 JSON 응답을 나란히 띄워봅니다.

In [ ]:
for r in results:
    if r["error"]:
        continue
    print(f"\n{'='*60}")
    print(f"  {r['model'].replace('-preview','')}  /  thinking={r['thinking']}")
    print(f"  {'='*60}")
    try:
        parsed = json.loads(r["response"])
        for k, v in parsed.items():
            print(f"    {k:25s}: {v}")
    except json.JSONDecodeError:
        print(r["response"][:300])

> **관찰 포인트** (조별로 토론하세요)
>
> 1. **Flash-Lite low**가 차량 대수를 환각(hallucination)으로 지어내지 않았나?
> 2. **Pro high**의 정체 원인 추정이 다른 조합 대비 얼마나 더 구체적인가?
> 3. JSON 스키마를 모든 조합이 정확히 따랐는가? 이탈한 조합은 어떤 패턴인가?
> 4. **가장 비싼 조합**과 **가장 싼 조합**의 비용 차이는 몇 배인가?

## 1-8. ITS 워크로드 매핑 — 의사결정 프레임

측정 결과를 바탕으로 ITS 도메인 워크로드별 권장 조합을 정리합니다.

In [ ]:
recommendations = pd.DataFrame([
    {
        "ITS 워크로드": "실시간 CCTV 이벤트 검지 (사고/낙하물/역주행)",
        "권장 모델": "3 Flash 또는 3.1 Flash-Lite",
        "Thinking": "low / minimal",
        "이유": "1초 내 응답 + 대량 처리, 단순 분류 태스크",
    },
    {
        "ITS 워크로드": "교차로 신호 최적화 시뮬레이션 분석",
        "권장 모델": "3.1 Pro",
        "Thinking": "high",
        "이유": "도로공학 다단계 추론, 호출 빈도 낮음",
    },
    {
        "ITS 워크로드": "VDS 시계열 이상 패턴 자동 분석",
        "권장 모델": "3 Flash",
        "Thinking": "medium",
        "이유": "패턴 인식 + 약간의 추론, 비용/성능 균형",
    },
    {
        "ITS 워크로드": "민원 텍스트 카테고리 자동 분류 (1만건+)",
        "권장 모델": "3.1 Flash-Lite",
        "Thinking": "low",
        "이유": "단순 분류, 대량 처리, 비용 최우선",
    },
    {
        "ITS 워크로드": "사고 영상 사후 분석 + 보고서 초안",
        "권장 모델": "3.1 Pro",
        "Thinking": "medium / high",
        "이유": "비디오 이해 + 인과 추론 + 문서화",
    },
    {
        "ITS 워크로드": "표준노드링크 데이터 자연어 질의",
        "권장 모델": "3 Flash",
        "Thinking": "low",
        "이유": "구조화된 데이터 단순 조회, 응답 속도 중요",
    },
])
recommendations

## 1-9. 도전 과제 (시간 여유가 있을 때)

다음 시나리오에서 **본인이라면 어떤 조합을 쓸지** 측정 결과를 근거로 제시하세요:

1. **시나리오 A**: 24시간 무인 운영되는 도로 CCTV에서 1분에 1번 정지영상을 분석해 사고 여부 판단. 월 호출 약 43만 건. 어떤 조합?

2. **시나리오 B**: 도로공사 주간 회의용 사고 통계 분석 리포트를 매주 월요일 1회 자동 생성. 입력 50K 토큰, 출력 약 5K 토큰. 어떤 조합?

3. **시나리오 C**: 시민 민원 챗봇 "이 도로 언제 뚫려요?". 평일 14시 피크에 분당 200건 호출. 어떤 조합?

각 시나리오에 대해 측정 결과를 인용하며 답하고, **월 추정 비용**을 계산해보세요.

## 1-10. 정리

- ✅ 9개 조합의 비용·성능을 정량 측정
- ✅ ITS 워크로드별 모델 선택 프레임 확보
- ✅ thinking 토큰이 비용에 미치는 영향 체감
- ✅ JSON 강제(`response_mime_type`) 사용 패턴 확보

**핵심 교훈**: 명시하지 않은 thinking_level은 `high`로 동작하며, 이건 비용 누수의 가장 흔한 원인입니다. 프로덕션 코드는 **반드시 명시적으로** thinking_level을 설정해야 합니다.

다음 핸즈온(`02_cctv_multimodal.ipynb`)에서는 멀티모달 입력을 본격적으로 다룹니다.